# Analitycal model of Google's MICROSERVICES DEMO

This notebook requires some json file to build the jackson network and solve the traffic equations. In particular we will need:

1. overall_avg_metrics-low_load.json collected in low load to estimate service rates for leaf services.
2. routing_count.json for building the routing matrix and solving the traffic equations.
3. overall_avg_metrics-normal_load.json to measure empirical metrics under experimental load for comparisons with model predictions.
4. upstream_response_time-low_load.json to estimate service rates for upstream nodes.
5. upstream_response_time-normal_load.json to measure the response time of upstream services under experimental arrival rates.

The notebook will be able to conclude whether with the currenct external arrival rate the system will be able to reach a steady state assuming M/M/1 queueing behavior for all nodes. In the case a steady state is reachable the notebook will compute staedy state performance metrics and compare them with actual measurements.


## Estimating service rates
Since we have not collected CPU consumption metrics (which would have allowed us to calculate the service rate $\mu$ by dividing throughput X with CPU utilization U), we will need to estimate service time by collecting the average response time in low load scenarios. However, just doing this is not enough, since the response time of upstream services also account for the response time of relied upon downstream services, given the way we collected the metric. Because of this we will have to rely on trace data to estimate average service time by looking at the difference between the start time of a span for an upstream service and the start time for the next downstream service span.

$R_i \approx S_i$, then $\mu_i \approx \frac{1}{R_i}$


In [ ]:
# load json data
import json
import numpy as np

with open("../jupyter_data/service_average_metrics-low_load.json", "r") as f:
    metrics_dict = json.load(f)

with open("../jupyter_data/routing_count.json","r") as f:
    routing_dict=json.load(f)

with open("../jupyter_data/upstream_response_time-low_load.json", "r") as f:
    upstream_resp_dict=json.load(f)

other_endpoints = [
    "GetAds",
    "ListRecommendations",
    "GetCart",
    "AddItem",
    "EmptyCart",
    "PlaceOrder",
    "Charge",
    "GetQuote",
    "ShipOrder",
    "SendOrderConfirmation",
    "GetSupportedCurrencies",
    "Convert",
    "ListProducts",
    "GetProduct",
]
services = [
    "frontend",
    "adservice",
    "recommendationservice",
    "cartservice",
    "checkoutservice",
    "paymentservice",
    "shippingservice",
    "emailservice",
    "currencyservice",
    "productcatalogservice",
]

metrics = [
    "requests_total",
    "requests_duration",
    "active_requests",
]

In [ ]:
# Compute approximated service rate and save it in a json file

service_rate = {service: 0 for service in services}

for service, metric_dict in metrics_dict.items():
    
    if service in ["frontend", "checkoutservice", "recommendationservice"]:
        R_i=upstream_resp_dict[service]
    else:
        R_i = metric_dict["requests_duration"][-1][0]

    mu_i = 1 / R_i
    service_rate[service] = mu_i


In [39]:
# Run to show estimated service rate for each microservice
print("Service rates for each service:")

for service, rate in service_rate.items():
    print(f"{service} - {rate}")

Service rates for each service:
frontend - 1081.9230000716905
adservice - 3954.8022598870025
recommendationservice - 3429.7237736846037
cartservice - 1096.1763411234535
checkoutservice - 3543.131423910748
paymentservice - 3285.714285714285
shippingservice - 11954.708939153048
emailservice - 3384.3227541689084
currencyservice - 4613.916500994038
productcatalogservice - 15342.389173555272


## Solving Traffic Equations

Given the routing count of departing requests on each node of the Queueing network (used to compute routing probabilities) and the external arrival rate $\gamma$ for the frontend, we can solve the Traffic Equations and obtain the aggregate arrival rate $\lambda_i$ for each service i.

$\lambda_i = \gamma_i + \sum_{j=0}^{N-1}q_{ji}\lambda_j$ 

In the matrix form we have:

If $\underline{\lambda}=[\lambda_1, \lambda_2, ..., \lambda_N]^T$ and $\underline{\gamma}=[\gamma_1, \gamma_2, ..., \gamma_N]^T$, then $\underline{\lambda}=\underline{\gamma}+Q^T\underline{\lambda}$ which implies $(I-Q^T)\underline{\lambda}=\underline{\gamma}$

In [40]:
# Run to build the external arrival rate vector
service_index_map = {
        "frontend": 0,
        "adservice": 1,
        "recommendationservice": 2,
        "cartservice": 4,
        "checkoutservice": 8,
        "paymentservice": 7,
        "shippingservice": 5,
        "emailservice": 9,
        "currencyservice": 6,
        "productcatalogservice": 3,
    }

function_map = {
    "GetAds": "adservice",
    "ListRecommendations": "recommendationservice",
    "GetCart": "cartservice",
    "AddItem": "cartservice",
    "EmptyCart": "cartservice",
    "PlaceOrder": "checkoutservice",
    "Charge": "paymentservice",
    "GetQuote": "shippingservice",
    "ShipOrder": "shippingservice",
    "SendOrderConfirmation": "emailservice",
    "GetSupportedCurrencies": "currencyservice",
    "Convert": "currencyservice",
    "ListProducts": "productcatalogservice",
    "GetProduct": "productcatalogservice",
}
# Set the value you see from the collected metrics on the frontend in the empirical experiment
EXTERNAL_ARRIVAL_RATE=11.4

gamma=[0.0 for _ in range(10)]
gamma[0]=EXTERNAL_ARRIVAL_RATE
gamma=np.array(gamma).reshape(-1,1)


In [41]:
# Run to show gamma
print("External arrival rates for each service:")

for key, value in service_index_map.items():
        print(f"{key} - {gamma[value][0]}")

External arrival rates for each service:
frontend - 11.4
adservice - 0.0
recommendationservice - 0.0
cartservice - 0.0
checkoutservice - 0.0
paymentservice - 0.0
shippingservice - 0.0
emailservice - 0.0
currencyservice - 0.0
productcatalogservice - 0.0


In [42]:
# Run to build the routing matrix
Q=np.array([[0.0 for _ in range(10)] for _ in range(10)])

for service in services:
    total=0
    for name, value in routing_dict[service].items():
        if name=="in":
            continue
        if name=="out":
            total+=value
            continue
        total+=value
        Q[service_index_map[service],service_index_map[function_map[name]]]+=value
    Q[service_index_map[service],:]=Q[service_index_map[service],:]/total

In [43]:
# Run to show Q
print("Full routing matrix:")

for key, value in service_index_map.items():
        print(f"{key} - {value}")
print()
print("\t", end="")
for i in range(10):
    print(f"{i}", end="\t")
print()
for i in range(10):
    print(i, end="\t")
    for j in range(10):
        print(f"{Q[i,j]:.3f}", end="\t")
    print()

Full routing matrix:
frontend - 0
adservice - 1
recommendationservice - 2
cartservice - 4
checkoutservice - 8
paymentservice - 7
shippingservice - 5
emailservice - 9
currencyservice - 6
productcatalogservice - 3

	0	1	2	3	4	5	6	7	8	9	
0	0.000	0.000	0.000	0.502	0.032	0.000	0.342	0.000	0.031	0.000	
1	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	
2	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	0.000	
3	0.000	0.088	0.000	0.671	0.054	0.040	0.147	0.000	0.000	0.000	
4	0.000	0.000	0.198	0.033	0.000	0.000	0.602	0.000	0.000	0.033	
5	0.000	0.000	0.000	0.000	0.125	0.000	0.875	0.000	0.000	0.000	
6	0.000	0.051	0.133	0.122	0.194	0.010	0.409	0.010	0.000	0.000	
7	0.000	0.000	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	
8	0.000	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	
9	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	


In [44]:
# Run to solve for lambda and print results

aggregate_arrival_rates = np.linalg.solve(np.eye(10) - Q.T, gamma)

print("Aggregate arrival rates for each service:")

for key, value in service_index_map.items():
        print(f"{key} - {aggregate_arrival_rates[value][0]}")

Aggregate arrival rates for each service:
frontend - 11.4
adservice - 6.440774050551979
recommendationservice - 7.129069519628317
cartservice - 10.693482801244617
checkoutservice - 0.3518008610033673
paymentservice - 0.3518008610033671
shippingservice - 2.823153318272877
emailservice - 0.3518008610033672
currencyservice - 34.90530241677677
productcatalogservice - 53.07674012190443


## Compute Performance Metrics

We assign different queueing models to the service nodes and analyze if the network can reach the steady state under current arrival rate. If it can then we compute performance metrics at steady state considering two different modeling scenarios:

- Mean queue length at M/M/1 node i: $L_i=\frac{\rho_i}{1-\rho_i}$, with $\rho_i=\frac{\lambda_i}{\mu_i}$
- Mean queue length at M/M/inf node i: $L_i=\rho_i$, with $\rho_i=\frac{\lambda_i}{\mu_i}$
- Mean queue length at M/M/c node i: $L_i=\frac{\rho_i}{1-\rho_i}*C(c_i,\frac{\lambda_i}{\mu_i})+c_i*\rho$, with $\rho_i=\frac{\lambda_i}{c_i*\mu_i}$ and $C(c,\frac{\lambda}{\mu})$ the Erlang's C formula.
- Mean system queue length: $L=L_1+L_2+...+L_N=\sum_{i=0}^{N-1}\frac{\rho_i}{1-\rho_i}$
- Mean system response time: $W=\frac{L}{\gamma}$, where $\gamma=\sum_{i=0}^{N-1}\gamma_i$
- Mean response time at node i: $W_i=\frac{L_i}{\lambda_i}$

In [45]:
# Run to compute utilization for every node and then print it
import math
mu=np.zeros((10,1),float)

for service in services:
        mu[service_index_map[service]]=service_rate[service]

rho=np.divide(aggregate_arrival_rates, mu)
print("Rho for each service::")
for key, value in service_index_map.items():
        print(f"{key} - {rho[value][0]}")

Rho for each service::
frontend - 0.010536794207392404
adservice - 0.0016285957242110018
recommendationservice - 0.0020786133199203532
cartservice - 0.009755257799383845
checkoutservice - 9.929094321177211e-05
paymentservice - 0.00010706982726189436
shippingservice - 0.00023615408226516707
emailservice - 0.00010395015090390198
currencyservice - 0.007565221956066317
productcatalogservice - 0.003459483364780599


In [46]:
# Run to define function to compute average queue length for M/M/1 servers
def M_M_1_queue_length(rho):
    return rho/(1.0-rho)

In [47]:
# Run to define function to compute average queue length for M/M/inf servers
def M_M_inf_queue_length(rho):
    return rho

In [48]:
# Run to define function to compute the Erlang's C formula
def Erlang_c(c, rho):
    rho_c=rho/c
    sum=0
    for k in range(c):
        sum+=rho**k/math.factorial(k)
    return 1/(1+(1-rho_c)*(math.factorial(c)/rho**c)*sum)

In [49]:
# Run to define function to compute average queue length for M/M/c servers
def M_M_c_queue_length(rho, c):
    rho_c=rho/c
    return rho_c*Erlang_c(c,rho)/(1-rho_c)+rho

In [50]:
# Run to check if the system can actually handle the load assuming the number of server to be equivalent to the cuncurrent behavior of the services.

steady_state=True

for service in services:
    if service in ["frontend", "checkoutservice", "adservice", "shippingservice", "cartservice", "productcatalogservice"]:
        continue
    elif service in ["currencyservice", "emailservice", "recommendationservice"]:
        if aggregate_arrival_rates[service_index_map[service]] > 10 * mu[service_index_map[service]]:
            print(f"Steady state does not exist, in {service} lambda is bigger than mu*c: {aggregate_arrival_rates[i,0]}>{mu[i,0]}")
            steady_state=False
            break
    else:
        if aggregate_arrival_rates[service_index_map[service]] > mu[service_index_map[service]]:
            print(f"Steady state does not exist, in {service} lambda is bigger than mu: {aggregate_arrival_rates[i,0]}>{mu[i,0]}")
            steady_state=False
            break

if steady_state:
    print("Steady state is reachable, we can now proceed to compute the steady state performance of the system.")

Steady state is reachable, we can now proceed to compute the steady state performance of the system.


In [51]:
# Run to heck if the system can actually handle the load assuming universal M/M/1 behavior
steady_state=True

for service in services:
    if aggregate_arrival_rates[service_index_map[service]] >  mu[service_index_map[service]]:
            print(f"Steady state does not exist, in {service} lambda is bigger than mu: {aggregate_arrival_rates[i,0]}>{mu[i,0]}")
            steady_state=False
            break

if steady_state:
    print("Steady state is reachable, we can now proceed to compute the steady state performance of the system.")

Steady state is reachable, we can now proceed to compute the steady state performance of the system.


In [52]:
# Run to compute mean queue length for every node and the mean system queue length given the second assumption (multiple types of queues), then print them.
L=np.zeros((10,1),float)
for service in services:
        if service in ["frontend", "checkoutservice", "adservice", "shippingservice", "cartservice", "productcatalogservice"]:
                L[service_index_map[service],0]=M_M_inf_queue_length(rho[service_index_map[service],0])
        elif service in ["currencyservice", "emailservice", "recommendationservice"]:
                L[service_index_map[service],0]=M_M_c_queue_length(rho[service_index_map[service],0],10)
        else:
                L[service_index_map[service],0]=M_M_1_queue_length(rho[service_index_map[service],0])

L_tot=np.sum(L)
print("Mean queue length for each service::")
for key, value in service_index_map.items():
        print(f"{key} - {L[value][0]}")
print(f"Mean number of requests in the system: ")
print(L_tot)

Mean queue length for each service::
frontend - 0.010536794207392404
adservice - 0.0016285957242110018
recommendationservice - 0.0020786133199203532
cartservice - 0.009755257799383845
checkoutservice - 9.929094321177211e-05
paymentservice - 0.00010708129243737862
shippingservice - 0.00023615408226516707
emailservice - 0.00010395015090390198
currencyservice - 0.007565221956066317
productcatalogservice - 0.003459483364780599
Mean number of requests in the system: 
0.03557044284057274


In [53]:
# Run to compute mean queue length for every node and the mean system queue length given the first assumption (all M/M/1 queues), then print them.
L=np.zeros((10,1),float)
for service in services:
    L[service_index_map[service],0]=M_M_1_queue_length(rho[service_index_map[service],0])
L_tot=np.sum(L)
print("Mean queue length for each service::")
for key, value in service_index_map.items():
        print(f"{key} - {L[value][0]}")
print(f"Mean number of requests in the system: ")
print(L_tot)

Mean queue length for each service::
frontend - 0.010649000534539255
adservice - 0.0016312523748537977
recommendationservice - 0.002082942952886858
cartservice - 0.009851360359364071
checkoutservice - 9.930080288215197e-05
paymentservice - 0.00010708129243737862
shippingservice - 0.00023620986418886657
emailservice - 0.00010396095766113898
currencyservice - 0.007622890817044116
productcatalogservice - 0.0034714929368465735
Mean number of requests in the system: 
0.035855492892704205


In [54]:
# Run to compute mean system response time and print the results
gamma_tot=np.sum(gamma)
W_tot=L_tot/gamma_tot
print("Mean system response time: ")
print(W_tot)

Mean system response time: 
0.0031452186747986145


In [55]:
# Run to compute response time at node i and show the results
W=np.divide(L,aggregate_arrival_rates)
print("Mean response time for each service:: ")
for key, value in service_index_map.items():
        print(f"{key} - {W[value][0]}")

Mean response time for each service:: 
frontend - 0.0009341228539069522
adservice - 0.00025326961667192753
recommendationservice - 0.0002921759911517113
cartservice - 0.0009212490020760561
checkoutservice - 0.00028226424062447505
paymentservice - 0.0003043804160455245
shippingservice - 8.366880489982489e-05
emailservice - 0.00029551081075990867
currencyservice - 0.0002183877602899746
productcatalogservice - 6.540516484006731e-05


## Comparisons with empirically collected data

Like previously said, the metrics we have collected for upstream services also comprehend downstream contributions. For example in the frontend service the mean duration is just the response time for the entire system, while the mean active requests represents the mean user number in the system. We will derive the corresponding empirical metrics in the following way:

- Empirical Average System Response Time: just the mean response time at the frontend.
- Empirical Average Number of Requests In The System: just the mean active requests number in the frontend.
- Empirical Average System Arrival Rate: just the mean number of arrival of requests per second in the frontend.
- Empirical Response Time At Service i: for leaf services they correspond to their measured local response time, while for upstream services we compute them from traces.
- Empirical Average Service Arrival Rates: we already got that as the local arrival rate at each service.
- Empirical Queue Length at Service i: estimated using Little's Law on the singular queues for upstream services, while for leaf services it corresponds to their mean active requests metric.


In [ ]:
# Run to see the empirical average system arrival rate at steady state
with open("../jupyter_data/service_average_metrics-normal_load.json", "r") as f:
    normal_metrics_dict=json.load(f)

with open("../jupyter_data/upstream_response_time-normal_load.json", "r") as f:
    normal_upstream_resp_dict=json.load(f)

print(f"Average system arrival rate: {normal_metrics_dict["frontend"]["requests_rate"]}")
print(f"System arrival rate used in the model: {np.sum(gamma)}")

Average system arrival rate: 11.373201086437303
System arrival rate used in the model: 11.4


In [ ]:
# Run to see the empirical average number of requests in the system

print(f"Average system queue length {normal_metrics_dict["frontend"]["active_requests"]-1][0]}")
print(f"Predicted system queue length: {L_tot}")

Average system queue length 0.6320848339262901
Predicted system queue length: 0.035855492892704205


In [ ]:
# Run to see the empirical average system response time

print(f"Average system response time {normal_metrics_dict["frontend"]["requests_duration"][-1][0]}")
print(f"Predicted system response time: {W_tot}")

Average system response time 0.038968862847142446
Predicted system response time: 0.0031452186747986145


In [ ]:
# Run to compute and show the empirical response time for each node i

service_rate = {service: 0 for service in services}

for service, metric_dict in normal_metrics_dict.items():
    
    if service in ["frontend", "checkoutservice", "recommendationservice"]:
        R_i=normal_upstream_resp_dict[service]
    else:
        R_i = metric_dict["requests_duration"][-1][0]

    service_rate[service] = R_i

print("Response time at each service node compared with model prediction")

W_emp=np.zeros(10,float)

for service, rsp_time in service_rate.items():
    print(f"{service}\t\t predicted response time: {W[service_index_map[service]][0]}, actual response time: {rsp_time}")
    W_emp[service_index_map[service]]=rsp_time

Response time at each service node compared with model prediction
frontend		 predicted response time: 0.0009471284485487625, actual response time: 0.0027293068999231502
adservice		 predicted response time: 1.8346656371955472e-05, actual response time: 0.0004398860398856139
recommendationservice		 predicted response time: 0.00029296286013391404, actual response time: 0.0010890177362889474
cartservice		 predicted response time: 0.0006019274830515247, actual response time: 0.001921702369993304
checkoutservice		 predicted response time: 0.00028230038799563066, actual response time: 0.0007768085308330237
paymentservice		 predicted response time: 0.00015386521984924478, actual response time: 0.000304761904761905
shippingservice		 predicted response time: 8.231865565247001e-05, actual response time: 0.0005019630292249423
emailservice		 predicted response time: 0.0003142476535601399, actual response time: 0.0004861661703279703
currencyservice		 predicted response time: 0.0002440874513019561, a

In [ ]:
# Run to show the empirical arrival rate for each node
lambda_emp=np.zeros(10,float)
print("Arrival rate at each service node compared with model prediction")
for service in services:
    print(f"{service}\t\t predicted arrival rate: {aggregate_arrival_rates[service_index_map[service]][0]}, actual arrival rate: {normal_metrics_dict[service]["requests_rate"][-1][0]}")
    lambda_emp[service_index_map[service]]=normal_metrics_dict[service]["requests_rate"][-1][0]

Arrival rate at each service node compared with model prediction
frontend		 predicted arrival rate: 26.1, actual arrival rate: 26.158146387398613
adservice		 predicted arrival rate: 14.745982694684788, actual arrival rate: 14.625001685321525
recommendationservice		 predicted arrival rate: 16.321817058096403, actual arrival rate: 16.046174219294695
cartservice		 predicted arrival rate: 24.4824474660074, actual arrival rate: 24.5500098836305
checkoutservice		 predicted arrival rate: 0.8054388133498146, actual arrival rate: 0.8750002851089771
paymentservice		 predicted arrival rate: 0.8054388133498144, actual arrival rate: 0.8749562518814508
shippingservice		 predicted arrival rate: 6.463535228677376, actual arrival rate: 6.558151970471388
emailservice		 predicted arrival rate: 0.8054388133498144, actual arrival rate: 0.8411924488786265
currencyservice		 predicted arrival rate: 79.91477132262047, actual arrival rate: 80.60547841283483
productcatalogservice		 predicted arrival rate: 121.51

In [ ]:
# Run to compute and show empirical queue length for each node

L_emp=np.zeros(10,float)

for service, metric_dict in normal_metrics_dict.items():
    
    if service in ["frontend", "checkoutservice", "recommendationservice"]:
        L_i = normal_metrics_dict[service]["requests_rate"][-1][0]*service_rate[service]
    else:
        L_i = metric_dict["active_requests"]

    service_rate[service] = L_i
    L_emp[service_index_map[service]]=L_i

print("Queue length at each service node compared with model prediction")

for service in services:
    print(f"{service}\t\t predicted queue length: {L[service_index_map[service]][0]}, actual queue length: {service_rate[service]}")

Queue length at each service node compared with model prediction
frontend		 predicted queue length: 0.0247200525071227, actual queue length: 0.07139360942432686
adservice		 predicted queue length: 0.0002705394773661838, actual queue length: 0.0
recommendationservice		 predicted queue length: 0.004781686207922429, actual queue length: 0.017474568324394375
cartservice		 predicted queue length: 0.014736657982155014, actual queue length: 0.0
checkoutservice		 predicted queue length: 0.000227375689515393, actual queue length: 0.0006797076859539814
paymentservice		 predicted queue length: 0.00012392902009118403, actual queue length: 0.0
shippingservice		 predicted queue length: 0.0005320695307871019, actual queue length: 0.0
emailservice		 predicted queue length: 0.0002531072571814427, actual queue length: 0.0
currencyservice		 predicted queue length: 0.019506192853517082, actual queue length: 0.0
productcatalogservice		 predicted queue length: 0.007715521656780042, actual queue length: 0.0
